# 02 – The Intern: Fine-Tuning (QLoRA)

**Operation Ledger-Mind: The Financial Intelligence**

**Model:** `finetune_model` in config (`unsloth/Llama-3.2-3B-Instruct`) | **Precision:** `4-bit NF4, FP16 compute` | **Strategy:** `QLoRA`

> Runs on a T4 (Colab or Kaggle free tier).
> Upload `config.yaml` (from `src/config/`) and `train.jsonl` to `/content/` before running.
> The base model is loaded in 4-bit and frozen; only the LoRA adapters are trained and saved.

In [3]:
# Setup
import sys
import os
from pathlib import Path
from dotenv import load_dotenv
import json
import torch

!pip install --upgrade torchao peft
!pip install -q trl peft accelerate transformers
!pip install -U bitsandbytes

sys.path.append(str(Path.cwd().parent))
load_dotenv()

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB")

CUDA available: True
GPU: Tesla T4
VRAM: 15.6GB


In [4]:
# Load config and training data
import yaml

with open("/content/config.yaml") as f:
    config = yaml.safe_load(f)

train_file = "/content/train.jsonl"

train_data = []
with open(train_file) as f:
    for line in f:
        train_data.append(json.loads(line))

print(f"Loaded {len(train_data)} training examples")

Loaded 1415 training examples


In [5]:
# Format for instruction tuning
from datasets import Dataset

# Same system prompt for training and inference
SYSTEM_PROMPT = "You are a financial analyst trained on Uber's 2024 Annual Report."

def format_instruction(example):
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": example['question']},
            {"role": "assistant", "content": example['answer']}
        ]
    }

formatted = [format_instruction(ex) for ex in train_data]
dataset = Dataset.from_list(formatted)
print(f"Formatted {len(dataset)} examples")

Formatted 1415 examples


In [6]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

model_id = config["finetune_model"]

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=bnb_config,               # 4-bit QLoRA
    dtype=torch.float16,
    attn_implementation="eager"
)

model.config.dtype = torch.float16

print(f"Model loaded in 4-bit: {model_id}")

W1119 09:26:28.159000 560 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1119 09:26:28.279000 560 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

... (3 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

Model loaded in 4-bit: unsloth/Llama-3.2-3B-Instruct


In [7]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = prepare_model_for_kbit_training(model)


In [8]:
# Training
output_dir = "/content/finetuned_model/"

training_args = SFTConfig(
    output_dir=str(output_dir),
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=10,
    optim="paged_adamw_8bit",
    logging_steps=10,
    save_steps=50,
    fp16=True,
    bf16=False,
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    peft_config=lora_config,
)

model = trainer.model  # the LoRA-wrapped model; cell 7 saves and queries this

for p in model.parameters():
    if p.requires_grad:
        p.data = p.data.float()

print({p.dtype for p in model.parameters() if p.requires_grad})  # should print {torch.float32}

print("Training...")
trainer.train()
print("Training complete!")


Tokenizing train dataset:   0%|          | 0/1415 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1415 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1415 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/1415 [00:00<?, ? examples/s]

{torch.float32}
Training...


Step,Training Loss
10,3.308689
20,1.513538
30,1.153119
40,1.094516
50,0.948985
60,1.017628
70,1.064628
80,0.959619
90,0.934066
100,1.021873


Training complete!


In [9]:
# Save adapters and create inference function
adapter_path = f"/content/finetuned_model/{config['adapter_name']}"
model.save_pretrained(str(adapter_path))
tokenizer.save_pretrained(str(adapter_path))

def query_intern(question, max_tokens=256):
    model.eval()

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=0.1,
            do_sample=True,
            use_cache=True
        )

    # ONLY decode new tokens
    generated = outputs[0][inputs["input_ids"].shape[-1]:]
    response = tokenizer.decode(generated, skip_special_tokens=True)

    return response.strip()

print("query_intern() function ready")
print("\nTesting...")
print(query_intern("What is Uber's mission?"))
print("\nThe Intern is ready!")

[transformers] Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


query_intern() function ready

Testing...


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Uber's mission is to reimagine the way the world moves, delivering mobility, logistics, and delivery solutions that improve lives and reduce carbon emissions.

The Intern is ready!
